<a href="https://colab.research.google.com/github/PedroK0/Intelig-ncia-Artificial-e-Programa-o-Para-Engenharia/blob/main/exercicio_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Exercício 24: Construção da Matriz de Admitância Nodal (Y_bus) e Solução Linear

Nesta seção, implementamos a criação estruturada da matriz de admitância de barramento ($Y_{bus}$) e resolvemos as equações de balanço nodal de correntes ($[I_{nodal}] = [Y_{bus}] \times [V_{nodal}]$) para determinar o perfil de tensões complexas de um sistema elétrico de 3 barras.
```

In [11]:
import numpy as np

def montar_ybus(n_barras: int, linhas: list, shunts: list = None) -> np.ndarray:
    """
    Monta de forma automatizada a matriz de admitância nodal Y_bus de um sistema elétrico,
    incluindo possíveis conexões shunt (derivações para a terra).

    Parâmetros:
        n_barras (int): Quantidade total de barras do sistema.
        linhas (list): Lista de tuplas no formato (barra_de, barra_para, Z_serie).
        shunts (list, opcional): Lista de tuplas no formato (barra, Z_shunt) conectadas à terra.
    Retorna:
        np.ndarray: Matriz complexa Y_bus de dimensões (N x N).
    """
    Y_bus = np.zeros((n_barras, n_barras), dtype=np.complex128)

    # Elementos de linha (série)
    for de, para, z_serie in linhas:
        k = de - 1
        m = para - 1
        y_serie = 1.0 / z_serie

        Y_bus[k, m] -= y_serie
        Y_bus[m, k] -= y_serie
        Y_bus[k, k] += y_serie
        Y_bus[m, m] += y_serie

    # Elementos de derivação para a terra (shunt) para remover a singularidade
    if shunts:
        for barra, z_shunt in shunts:
            k = barra - 1
            y_shunt = 1.0 / z_shunt
            Y_bus[k, k] += y_shunt

    return Y_bus

# 1. Definição da rede de 3 barras
n_barras = 3
conexoes = [
    (1, 2, 0.20 + 0.40j),  # Linha 1–2
    (2, 3, 0.10 + 0.20j),  # Linha 2–3
    (1, 3, 0.20 + 0.40j)   # Linha 1–3
]

# Adicionamos uma referência à terra na Barra 1 (Z_shunt = 1.0 Ohm) para ancorar o potencial de referência
shunts = [
    (1, 1.0 + 0.0j)
]

# 2. Geração da Matriz Y_bus não-singular
Y_bus = montar_ybus(n_barras, conexoes, shunts)

# 3. Vetor de injeção de correntes conhecidas nas barras
I_nodal = np.array([
    50.0 + 0.0j,
    -25.0 - 10.0j,
    -25.0 + 10.0j
], dtype=np.complex128)

# 4. Solução do sistema linear Y_bus * V_nodal = I_nodal
V_nodal = np.linalg.solve(Y_bus, I_nodal)

# --- Exibição formatada para apresentação ---
print("=== MATRIZ DE ADMITÂNCIA NODAL Y_bus CORRIGIDA (Com shunt de referência) ===\n")
np.set_printoptions(precision=4, suppress=True)
print(Y_bus)

print("\n=== PERFIL DE TENSÕES NODAIS CALCULADOS ===\n")
fases = [1, 2, 3]
for idx in range(n_barras):
    mag = np.abs(V_nodal[idx])
    ang = np.angle(V_nodal[idx], deg=True)
    print(f"Barra {fases[idx]}: {V_nodal[idx]:.2f} V  |  Magnitude: {mag:.2f} V  |  Ângulo: {ang:.1f}°")

=== MATRIZ DE ADMITÂNCIA NODAL Y_bus CORRIGIDA (Com shunt de referência) ===

[[ 3.-4.j -1.+2.j -1.+2.j]
 [-1.+2.j  3.-6.j -2.+4.j]
 [-1.+2.j -2.+4.j  3.-6.j]]

=== PERFIL DE TENSÕES NODAIS CALCULADOS ===

Barra 1: 0.00-0.00j V  |  Magnitude: 0.00 V  |  Ângulo: -23.9°
Barra 2: -4.20-10.40j V  |  Magnitude: 11.22 V  |  Ângulo: -112.0°
Barra 3: -5.80-9.60j V  |  Magnitude: 11.22 V  |  Ângulo: -121.1°


```markdown
## Explicação do Código para a Apresentação

* **`montar_ybus`**: Esta função automatiza o processo clássico de montagem de redes de energia elétrica. Ela calcula a admitância série ($y = 1/Z$) de cada condutor e, de forma indexada, acumula esses termos nas posições de acoplamento mútuo (fora da diagonal com sinal invertido, $Y_{km} = -y$) e auto-admitância própria de cada barra (na diagonal principal, $Y_{kk} = \sum y$).
* **`np.linalg.solve(Y_bus, I_nodal)`**: Métodos que invertem diretamente matrizes ($Y^{-1}$) são numericamente instáveis e computacionalmente ineficientes. Usar o resolvedor nativo do NumPy resolve a equação linear $[Y_{bus}][V] = [I]$ por meio de fatorações matriciais diretas e seguras, garantindo a escala rápida de cálculo para sistemas com milhares de barras.
```